# Data Preprocessing

The purpose of this notebook is to handle all the preprocessing. Data is exported to `../data/notebooks_exports/`.

_structure work in progress_

## libraries

In [ ]:
import json
import pprint
import random
import pandas as pd
import numpy as np

from pathlib import Path

## settings

In [ ]:
DATA_DIR = Path("../data")
EXPORT_DIR = DATA_DIR / "notebooks_exports"

INPUT_DATA = DATA_DIR / "thesis_data_1778418853489.json"

TICKS_PER_SECOND = 20 # amount of ticks (steps) per second in minecraft

MIN_WINDOW_SIZE = 60 #size for windows to be considered windows
RANDOM_WINDOW_SIZE = 60 # size for random windows in ticks
RANDOM_SEED = 18 # to make the random windows reproducable

IDLE_THRESHOLD = 0.01 # distance moved that is still considered no movement
MAX_IDLE_TICKS = 3 # consecutive ticks without movement needed for idle
MIN_BLOCKS = 5 # min amount of blocks in window to make data meaningful

## importing data

In [ ]:
# load the json
with open(INPUT_DATA, "r", encoding="utf-8") as f:
    data = json.load(f)

# sanity checks
print(f"data set: {len(data)} replays")
print(f"replay structure: {list(data[0].keys())}")

## blocks / clicks & movement

helper functions:

In [ ]:

def movement_per_tick(steps):
    """moved distance between each tick, returns an (x-1, ) 1D array, where x is the number of input ticks"""
    positions = np.array([[ # (x, 3) array with the (x,y,z) positions
        step["playerPosition"]["x"],
        step["playerPosition"]["y"],
        step["playerPosition"]["z"]
    ] for step in steps])
    deltas = np.diff(positions, axis=0) # (x-1, 3) array with the differences between each row
    movement = np.sqrt((deltas ** 2).sum(axis=1)) # (x-1, ) 1D array with calculated distance in each row

    return movement


def longest_active_window(movement):
    """finds the longest active window in movement, returns a slice or none if too short"""
    is_idle = movement < IDLE_THRESHOLD
    keep = np.ones(len(movement), dtype=bool)
    idle_ticks_combo = 0

    for i in range(len(movement)):
        if is_idle[i]:
            idle_ticks_combo += 1
        else:
            idle_ticks_combo = 0

        if idle_ticks_combo > MAX_IDLE_TICKS:
            # drop the whole idle part
            keep[i - idle_ticks_combo + 1 : i+1] = False

    best_start, best_length = 0, 0
    current_start, current_length = 0, 0

    for i in range(len(keep)):
        if keep[i]:
            # if keep and no active window, we start new window
            if current_length == 0:
                current_start = i
            current_length += 1
            # if current window is better, we put it as best window
            if current_length > best_length:
                best_start, best_length = current_start, current_length #
        else:
            # if not keep, we drop the window by setting length to 0
            current_length = 0

    if best_length < MIN_WINDOW_SIZE:
        return None

    return slice(best_start, best_start + best_length)


def replay_features(item):
    """calculates blocks/tick and average movement speed in replay, returns as dict"""
    # drop replays that are too short
    steps = item["stepData"]
    if len(steps) < MIN_WINDOW_SIZE:
        return None

    # drop replays with no viable window
    movement = movement_per_tick(steps)
    window = longest_active_window(movement)
    if window is None:
        return None

    # count clicks/block in window
    # use [1:] because movement of ticks 0-1 belong to step 1
    clicks = np.array([step["clicks"] for step in steps[1:]])[window].sum()
    blocks = np.array([len(step["blocksChanged"]) for step in steps[1:]])[window].sum()

    # drop if too few blocks placed in window
    if blocks < MIN_BLOCKS or clicks == 0:
        return None

    return {
        "fastBuilderMode": item["fastBuilderMode"],
        "idAnonymous": item["idAnonymous"],
        "blocks_per_click": blocks / clicks,
        "avg_speed": movement[window].mean() * TICKS_PER_SECOND,
        "window_share": len(movement[window]) / len(movement),
    }

building dataframe from whole dataset

In [ ]:
rows = []
for item in data:
    row = replay_features(item)
    if row is not None:
        rows.append(row)

df_speed_to_block_per_click = pd.DataFrame(rows)

print(f"dataframe holds {len(df_speed_to_block_per_click)} out of {len(data)} replays")
print(df_speed_to_block_per_click.head().head())


exporting the dataframe

In [ ]:
EXPORT_DIR.mkdir(exist_ok=True)
df_speed_to_block_per_click.to_csv(EXPORT_DIR / "speed_to_block_per_click.csv", index=False)